# DEAP Emotion Recognition — MMCAT
**MultiModal Cross-Attention Transformer** on EEG + PPG + GSR

| Modality | Features | Description |
|----------|----------|-------------|
| EEG (ch 0–31) | 160 | Differential Entropy per band x channel |
| PPG (ch 38)   |  10 | HRV: SDNN, RMSSD, pNN50, LF/HF, etc. |
| GSR (ch 36)   |   8 | EDA: SCL mean/std, SCR peaks, slope |

**Protocols:** Subject-Dependent (8-fold CV) · Subject-Independent (LOSO)

## 0. Setup

In [ ]:
import sys, os
from pathlib import Path

# Make sure the notebook finds the project modules
PROJECT_DIR = Path(".").resolve()
sys.path.insert(0, str(PROJECT_DIR))
os.chdir(PROJECT_DIR)
print("Working dir:", PROJECT_DIR)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {device}")
if device.type == "cuda":
    print(f"GPU     : {torch.cuda.get_device_name(0)}")

## 1. Feature Extraction
Первый запуск: ~5 мин на 32 субъекта. Повторный — мгновенный (кэш pkl).

In [ ]:
# ── Конфигурация ──────────────────────────────────────────────────────────────
from config import N_SUBJECTS

SUBJECT_IDS = list(range(1, N_SUBJECTS + 1))   # 1–32
USE_CACHE   = True    # False = пересчитать признаки заново

print(f"Subjects: {len(SUBJECT_IDS)}  ({SUBJECT_IDS[0]}–{SUBJECT_IDS[-1]})")

In [ ]:
import time, pickle
from config import CACHE_DIR, SFREQ, EEG_WINDOW_SEC
from data.loader import load_all_subjects
from features.eeg_features import extract_de_subject
from features.peripheral_features import extract_ppg_subject, extract_gsr_subject

CACHE_DIR.mkdir(parents=True, exist_ok=True)
cache_path = CACHE_DIR / f"features_win_s{'_'.join(str(s) for s in SUBJECT_IDS)}.pkl"

N_WINDOWS = int(60 / EEG_WINDOW_SEC)  # 60 windows per 60-s trial

if USE_CACHE and cache_path.exists():
    print(f"Loading from cache: {cache_path.name}")
    with open(cache_path, "rb") as f:
        features, labels = pickle.load(f)
else:
    t0 = time.time()
    print("Loading raw signals ...")
    signals, labels = load_all_subjects(SUBJECT_IDS, verbose=True)

    print("Extracting features (window-based) ...")
    features = {}
    for sid in SUBJECT_IDS:
        subj = signals[sid]
        eeg_feats, groups = extract_de_subject(subj["eeg"])
        features[sid] = {
            "eeg":    eeg_feats,
            "ppg":    extract_ppg_subject(subj["ppg"],    n_windows=N_WINDOWS),
            "gsr":    extract_gsr_subject(subj["gsr"],    n_windows=N_WINDOWS),
            "groups": groups,
        }
        print(f"  s{sid:02d} done", end="")

    with open(cache_path, "wb") as f:
        pickle.dump((features, labels), f)
    print(f"
Done in {time.time()-t0:.1f}s  |  saved to {cache_path.name}")

sid0 = SUBJECT_IDS[0]
n_total = len(features[sid0]["eeg"])
print(f"EEG  : {features[sid0]['eeg'].shape}  ({n_total} = 40 trials x {N_WINDOWS} windows)")
print(f"PPG  : {features[sid0]['ppg'].shape}")
print(f"GSR  : {features[sid0]['gsr'].shape}")
print(f"Labels: {labels[sid0].shape}  (per-trial, 40 x 2)")


### Label distribution

In [ ]:
all_lbl = np.concatenate(list(labels.values()))   # (32*40, 2)

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for i, (ax, name) in enumerate(zip(axes, ["Valence", "Arousal"])):
    counts = np.bincount(all_lbl[:, i])
    ax.bar(["Low", "High"], counts, color=["#5B9BD5", "#ED7D31"])
    ax.set_title(f"{name} distribution")
    ax.set_ylabel("Trials")
    for j, c in enumerate(counts):
        ax.text(j, c + 5, str(c), ha="center")
plt.suptitle(f"DEAP  |  {len(SUBJECT_IDS)} subjects x 40 trials", y=1.02)
plt.tight_layout()
plt.show()

## 2. Subject-Dependent (SD) — 8-fold CV
Обучает отдельную модель per субъект, 8 фолдов (5 трайлов каждый).  
**Время:** ~25–40 мин на RTX 4070 Ti для 32 субъектов.

In [ ]:
from training.trainer import run_subject_dependent, print_results

print("Starting SD training ...")
t_sd = time.time()
sd_results = run_subject_dependent(features, labels, device)
print(f"\nSD done in {time.time()-t_sd:.1f}s")
print_results(sd_results)

### SD results per subject

In [ ]:
sd_subj = sd_results["subjects"]
sids = sorted(sd_subj.keys())
val_accs = [sd_subj[s]["mean"]["valence_acc"] for s in sids]
ar_accs  = [sd_subj[s]["mean"]["arousal_acc"]  for s in sids]

x = np.arange(len(sids))
w = 0.35
fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(x - w/2, val_accs, w, label="Valence",  color="#5B9BD5")
ax.bar(x + w/2, ar_accs,  w, label="Arousal",  color="#ED7D31")
ax.axhline(np.mean(val_accs), color="#5B9BD5", linestyle="--", lw=1.5,
           label=f"Val mean {np.mean(val_accs):.1f}%")
ax.axhline(np.mean(ar_accs),  color="#ED7D31", linestyle="--", lw=1.5,
           label=f"Ar  mean {np.mean(ar_accs):.1f}%")
ax.set_xticks(x)
ax.set_xticklabels([f"s{s:02d}" for s in sids], rotation=45)
ax.set_ylabel("Accuracy (%)")
ax.set_title("SD: per-subject accuracy (8-fold CV mean)")
ax.legend()
ax.set_ylim(40, 105)
plt.tight_layout()
plt.show()

## 3. Subject-Independent (LOSO)
Leave-one-subject-out: обучение на 31, тест на 1.  
**Время:** ~5–7 ч на RTX 4070 Ti.  
Запускай если хочешь честную cross-subject оценку.

In [ ]:
RUN_LOSO = False   # <- set True to run LOSO (~5-7h)

if RUN_LOSO:
    from training.trainer import run_loso
    print("Starting LOSO training ...")
    t_loso = time.time()
    loso_results = run_loso(features, labels, device)
    print(f"LOSO done in {time.time()-t_loso:.1f}s")
    print_results(loso_results)
else:
    print("LOSO skipped. Set RUN_LOSO = True to run.")
    loso_results = None


### LOSO results per subject (если запускали)

In [ ]:
if RUN_LOSO:
    loso_subj = loso_results["subjects"]
    sids_l = sorted(loso_subj.keys())
    val_l = [loso_subj[s]["valence_acc"] for s in sids_l]
    ar_l  = [loso_subj[s]["arousal_acc"]  for s in sids_l]

    x = np.arange(len(sids_l))
    fig, ax = plt.subplots(figsize=(14, 4))
    ax.bar(x - w/2, val_l, w, label="Valence", color="#5B9BD5")
    ax.bar(x + w/2, ar_l,  w, label="Arousal", color="#ED7D31")
    ax.axhline(np.mean(val_l), color="#5B9BD5", linestyle="--", lw=1.5,
               label=f"Val mean {np.mean(val_l):.1f}%")
    ax.axhline(np.mean(ar_l),  color="#ED7D31", linestyle="--", lw=1.5,
               label=f"Ar  mean {np.mean(ar_l):.1f}%")
    ax.set_xticks(x)
    ax.set_xticklabels([f"s{s:02d}" for s in sids_l], rotation=45)
    ax.set_ylabel("Accuracy (%)")
    ax.set_title("LOSO: per-subject accuracy")
    ax.legend()
    ax.set_ylim(30, 105)
    plt.tight_layout()
    plt.show()

## 4. Summary table

In [ ]:
import pandas as pd

rows = []
agg = sd_results["aggregate"]
rows.append({
    "Protocol": "SD (8-fold, window-based)",
    "Valence Acc": f"{agg['valence_acc'][0]:.2f} +/- {agg['valence_acc'][1]:.2f}",
    "Arousal Acc": f"{agg['arousal_acc'][0]:.2f} +/- {agg['arousal_acc'][1]:.2f}",
    "Valence F1":  f"{agg['valence_f1'][0]:.2f} +/- {agg['valence_f1'][1]:.2f}",
    "Arousal F1":  f"{agg['arousal_f1'][0]:.2f} +/- {agg['arousal_f1'][1]:.2f}",
})

if loso_results is not None:
    agg_l = loso_results["aggregate"]
    rows.append({
        "Protocol": "LOSO",
        "Valence Acc": f"{agg_l['valence_acc'][0]:.2f} +/- {agg_l['valence_acc'][1]:.2f}",
        "Arousal Acc": f"{agg_l['arousal_acc'][0]:.2f} +/- {agg_l['arousal_acc'][1]:.2f}",
        "Valence F1":  f"{agg_l['valence_f1'][0]:.2f} +/- {agg_l['valence_f1'][1]:.2f}",
        "Arousal F1":  f"{agg_l['arousal_f1'][0]:.2f} +/- {agg_l['arousal_f1'][1]:.2f}",
    })

rows.append({
    "Protocol": "MS-MPHAN 2024 (SD, ref)",
    "Valence Acc": "93.75", "Arousal Acc": "93.10",
    "Valence F1": "-",      "Arousal F1": "-",
})

df = pd.DataFrame(rows).set_index("Protocol")
df


## 5. Save results

In [ ]:
import json, time as _time
from config import RESULTS_DIR
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

def _serial(obj):
    if isinstance(obj, (np.floating, float)):  return float(obj)
    if isinstance(obj, (np.integer, int)):     return int(obj)
    if isinstance(obj, np.ndarray):            return obj.tolist()
    if isinstance(obj, dict):  return {k: _serial(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)): return [_serial(v) for v in obj]
    return obj

out = {"sd": sd_results}
if RUN_LOSO:
    out["loso"] = loso_results

ts = _time.strftime("%Y%m%d_%H%M%S")
out_path = RESULTS_DIR / f"results_{ts}.json"
with open(out_path, "w") as f:
    json.dump(_serial(out), f, indent=2)
print(f"Saved: {out_path}")